In [32]:
import math
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.style as style
import numpy as np
import pandas as pd

import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway
import datetime
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import VotingRegressor
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [33]:
# 데이터 시각화
import matplotlib.pyplot as plt
import matplotlib

# 맑은 고딕 적용
matplotlib.rc("font", family = "AppleGothic")
# 음수 표시
matplotlib.rc("axes", unicode_minus = False)

In [34]:
df = pd.read_csv("df_merged.csv", encoding="cp949")
df.head()

,idUser,idOrder,OrderDT,ItemCode,Price,DeliveryDT,OrderYear,OrderMonth,OrderDay,OrderHour,...,ItemSmallCode,ItemSmallName,ItemName,PriceMin,PriceMax,Gender,Age,FamilyCount,MemberYN,AgeGroup
0,U10001,U10001-O2023-1002,2023-01-06 17:08:51,L4-M17-S0530-1024,33310,2023-01-07 06:24:00,2023,1,6,17,...,S0530,전복,완도 활전복 1kg 중 22-25미,33160.0,37070.0,여성,26,2,Y,20
1,U10001,U10001-O2023-1002,2023-01-06 17:08:51,L1-M21-S0540-1082,3780,2023-01-07 06:24:00,2023,1,6,17,...,S0540,즉석,동원 양반 차돌된장찌개 (460G),3690.0,3970.0,여성,26,2,Y,20
2,U10001,U10001-O2023-1002,2023-01-06 17:08:51,L1-M15-S0140-1311,22520,2023-01-07 06:24:00,2023,1,6,17,...,S0140,냉동,오뚜기 듬뿍 새우볶음밥450g (2인분) x 5봉지 /,22150.0,23150.0,여성,26,2,Y,20
3,U10001,U10001-O2023-1002,2023-01-06 17:08:51,L4-M12-S0350-1035,21630,2023-01-07 06:24:00,2023,1,6,17,...,S0350,사과,[산지직송] 새콤달콤 부사 사과 5kg (13과내),20810.0,23030.0,여성,26,2,Y,20
4,U10001,U10001-O2023-1003,2023-01-13 16:50:14,L4-M12-S0640-1057,11700,2023-01-14 06:28:00,2023,1,13,16,...,S0640,토마토,스테비아 방울 토마토 라루 토망고 1kg,11640.0,13020.0,여성,26,2,Y,20


In [35]:
df = df.dropna()

In [36]:
df.isnull().sum()

idUser              0
idOrder             0
OrderDT             0
ItemCode            0
Price               0
DeliveryDT          0
OrderYear           0
OrderMonth          0
OrderDay            0
OrderHour           0
OrderMinute         0
OrderWeekday        0
DeliveryYear        0
DeliveryMonth       0
DeliveryDay         0
DeliveryHour        0
DeliveryMinute      0
DeliveryWeekday     0
DeliveryDeadline    0
IsLate              0
ItemLargeCode       0
ItemLargeName       0
ItemMiddleCode      0
ItemMiddleName      0
ItemSmallCode       0
ItemSmallName       0
ItemName            0
PriceMin            0
PriceMax            0
Gender              0
Age                 0
FamilyCount         0
MemberYN            0
AgeGroup            0
dtype: int64

In [37]:
# 고객별 평균 객단가 계산
order_total = df.groupby('idOrder')['Price'].sum().reset_index()
order_total.columns = ['idOrder', 'order_total']

customer_avg = order_total.merge(df[['idOrder', 'idUser']].drop_duplicates(), on='idOrder')
avg_order_value = customer_avg.groupby('idUser')['order_total'].mean()

df['avg_order_value'] = df['idUser'].map(avg_order_value)
df['avg_order_value'].describe()

count    854101.000000
mean      77347.699249
std        6364.836365
min       55575.750000
25%       72985.131579
50%       77385.806452
75%       81352.777778
max      111365.200000
Name: avg_order_value, dtype: float64

In [38]:
# 검증: Null값 확인
print(f"Null값 개수: {df['avg_order_value'].isnull().sum()}")
print(f"\n고객별 평균 객단가 샘플:")
print(df[['idUser', 'avg_order_value']].drop_duplicates().head(10))

Null값 개수: 0

고객별 평균 객단가 샘플:
      idUser  avg_order_value
0     U10001     78478.115942
347   U10002     74466.981132
617   U10003     89794.339623
906   U10004     75709.696970
1067  U10005     71330.384615
1438  U10006     80902.142857
1789  U10007     80123.802817
2165  U10008     82596.769231
2492  U10009     82170.800000
2616  U10010     69648.333333


In [39]:
# Step 1: datetime 변환 및 행 단위 파생 플래그
df['OrderDT'] = pd.to_datetime(df['OrderDT'])
df['DeliveryDT'] = pd.to_datetime(df['DeliveryDT'])
df['is_weekend'] = df['OrderWeekday'].isin(['Saturday', 'Sunday']).astype(int)
df['is_night'] = ((df['OrderHour'] >= 22) | (df['OrderHour'] <= 5)).astype(int)

# 신선식품 범주 정의
fresh_cats = ['수산', '축산', '청과', '신선', '쌀', '농산']
df['is_fresh'] = df['ItemLargeName'].apply(lambda x: any(c in str(x) for c in fresh_cats)).astype(int)

# Step 2: 주문 단위 집계
order_agg = df.groupby(['idUser', 'idOrder']).agg(
    order_amount=('Price', 'sum'),
    item_count=('Price', 'count'),
    is_weekend=('is_weekend', 'first'),
    is_night=('is_night', 'first'),
    order_dt=('OrderDT', 'first'),
).reset_index()

# 주문 간격 계산
order_agg = order_agg.sort_values(['idUser', 'order_dt'])
order_agg['prev_dt'] = order_agg.groupby('idUser')['order_dt'].shift(1)
order_agg['interval_days'] = (order_agg['order_dt'] - order_agg['prev_dt']).dt.days

# Step 3: 고객 단위 집계
df_model = order_agg.groupby('idUser').agg(
    total_order_count=('idOrder', 'count'),
    avg_order_value=('order_amount', 'mean'),
    total_spend=('order_amount', 'sum'),
    avg_items_per_order=('item_count', 'mean'),
    weekend_order_ratio=('is_weekend', 'mean'),
    night_order_ratio=('is_night', 'mean'),
    order_interval_days=('interval_days', 'mean'),
).reset_index()

# Step 4: 카테고리 및 배송 파생변수
cust_cat = df.groupby('idUser').agg(
    category_diversity=('ItemLargeCode', 'nunique'),
    fresh_food_ratio=('is_fresh', 'mean'),
    delay_rate=('IsLate', 'mean'),
).reset_index()

# Step 5: 인구통계
cust_demo = df.groupby('idUser').agg(
    gender=('Gender', 'first'),
    age=('Age', 'first'),
    family_count=('FamilyCount', 'first'),
    member_yn=('MemberYN', 'first'),
).reset_index()

# 최종 merge
df_model = df_model.merge(cust_cat, on='idUser').merge(cust_demo, on='idUser')

print(f"df_model shape: {df_model.shape}")
print(f"\n컬럼: {df_model.columns.tolist()}")
print(f"\nNull값 확인:")
print(df_model.isnull().sum())

df_model shape: (3000, 15)

컬럼: ['idUser', 'total_order_count', 'avg_order_value', 'total_spend', 'avg_items_per_order', 'weekend_order_ratio', 'night_order_ratio', 'order_interval_days', 'category_diversity', 'fresh_food_ratio', 'delay_rate', 'gender', 'age', 'family_count', 'member_yn']

Null값 확인:
idUser                 0
total_order_count      0
avg_order_value        0
total_spend            0
avg_items_per_order    0
weekend_order_ratio    0
night_order_ratio      0
order_interval_days    0
category_diversity     0
fresh_food_ratio       0
delay_rate             0
gender                 0
age                    0
family_count           0
member_yn              0
dtype: int64


In [40]:
# df_model 샘플 및 기본 통계
print("df_model 컬럼:")
print(df_model.columns.tolist())
print(f"\ndf_model shape: {df_model.shape}")
print("\ndf_model 처음 10행:")
print(df_model.head(10))
print("\n기본 통계:")
df_model.describe()

df_model 컬럼:
['idUser', 'total_order_count', 'avg_order_value', 'total_spend', 'avg_items_per_order', 'weekend_order_ratio', 'night_order_ratio', 'order_interval_days', 'category_diversity', 'fresh_food_ratio', 'delay_rate', 'gender', 'age', 'family_count', 'member_yn']

df_model shape: (3000, 15)

df_model 처음 10행:
   idUser  total_order_count  avg_order_value  total_spend  \
0  U10001                 69     78478.115942      5414990   
1  U10002                 53     74466.981132      3946750   
2  U10003                 53     89794.339623      4759100   
3  U10004                 33     75709.696970      2498420   
4  U10005                 78     71330.384615      5563770   
5  U10006                 70     80902.142857      5663150   
6  U10007                 71     80123.802817      5688790   
7  U10008                 65     82596.769231      5368790   
8  U10009                 25     82170.800000      2054270   
9  U10010                 30     69648.333333      2089450   



,total_order_count,avg_order_value,total_spend,avg_items_per_order,weekend_order_ratio,night_order_ratio,order_interval_days,category_diversity,fresh_food_ratio,delay_rate,age,family_count
count,3000.000000,3000.000000,3.000000e+03,3000.000000,3000.000000,3000.000000,3000.000000,3000.00000,3000.000000,3000.000000,3000.000000,3000.000000
mean,57.009667,77429.636837,4.388288e+06,5.016592,0.183263,0.109869,17.378925,3.72100,0.482271,0.064913,36.785667,1.622667
std,12.990041,6725.503464,9.943655e+05,0.351588,0.086459,0.043134,3.276888,0.46393,0.024163,0.038283,10.949141,0.971563
min,9.000000,55575.750000,7.923600e+05,3.971831,0.000000,0.000000,10.733333,2.00000,0.370000,0.000000,21.000000,0.000000
25%,49.000000,72801.561668,3.739295e+06,4.780000,0.123077,0.080000,15.261905,3.00000,0.467122,0.037401,29.000000,1.000000
50%,60.000000,77351.911765,4.459500e+06,5.022739,0.177419,0.107143,16.770492,4.00000,0.482713,0.060911,35.000000,2.000000
75%,66.000000,81676.307998,5.081795e+06,5.235294,0.235294,0.136364,18.716270,4.00000,0.497899,0.088710,43.000000,2.000000
max,90.000000,111365.200000,7.432080e+06,6.640000,0.642857,0.312500,45.304348,4.00000,0.647059,0.267974,69.000000,3.000000
